# NarVid Vietnamese Fine-tuning (Notebook)

Notebook này chuyển luồng train/eval từ file train_narvid_vn.py sang dạng ipynb.

Mục tiêu:
- Train NarVid theo quy trình thông thường (1 GPU, không DDP).
- Load class/hàm từ các file có sẵn trong source và narvid.
- Eval và so sánh với baseline trong source/narvid_baseline_results.json.

Ghi chú: notebook chỉ cung cấp code, bạn chủ động chạy từng cell khi cần.

In [1]:
import json
import os
import random
import sys
import time
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch

cwd = Path.cwd().resolve()
if (cwd / 'narvid').exists():
    BASE_DIR = cwd
elif (cwd.parent / 'narvid').exists():
    BASE_DIR = cwd.parent
else:
    raise RuntimeError(f'Cannot locate narvid folder from cwd={cwd}')

SOURCE_DIR = BASE_DIR / 'source'
NARVID_DIR = BASE_DIR / 'narvid'

if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
if str(NARVID_DIR) not in sys.path:
    sys.path.insert(0, str(NARVID_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('BASE_DIR :', BASE_DIR)
print('DEVICE   :', DEVICE)
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))

BASE_DIR : /media/urlab/KINGSTON/aic
DEVICE   : cuda
GPU      : NVIDIA GeForce RTX 5060 Ti


In [2]:
from dataloader_vietnamese_narvid import dataloader_vietnamese_train, dataloader_vietnamese_test

from dataloaders.data_dataloaders import DATALOADER_DICT
from metrics import compute_metrics, tensor_text_to_video_metrics, tensor_video_to_text_sim
from modules.file_utils import PYTORCH_PRETRAINED_BERT_CACHE
from modules.modeling_narvid import NarVid
from modules.optimization import BertAdam
from modules.tokenization_clip import SimpleTokenizer as ClipTokenizer
from modules.until_module import AllGather
from util import get_logger

def _safe_allgather_forward(ctx, tensor, args):
    world_size = int(getattr(args, 'world_size', 1))
    rank = int(getattr(args, 'rank', 0))
    dist_ready = torch.distributed.is_available() and torch.distributed.is_initialized()

    if world_size <= 1 or not dist_ready:
        ctx.rank = 0
        ctx.batch_size = tensor.shape[0]
        return tensor

    output = [torch.empty_like(tensor) for _ in range(world_size)]
    torch.distributed.all_gather(output, tensor)
    ctx.rank = rank
    ctx.batch_size = tensor.shape[0]
    return torch.cat(output, dim=0)

def _patch_safe_barrier():
    if not hasattr(torch.distributed, 'barrier'):
        return
    if getattr(torch.distributed, '_narvid_safe_barrier_patched', False):
        return

    _orig_barrier = torch.distributed.barrier

    def _safe_barrier(*args, **kwargs):
        if torch.distributed.is_available() and torch.distributed.is_initialized():
            return _orig_barrier(*args, **kwargs)
        return None

    torch.distributed.barrier = _safe_barrier
    torch.distributed._narvid_safe_barrier_patched = True

# Patch runtime for notebook / single-GPU execution.
AllGather.forward = staticmethod(_safe_allgather_forward)
_patch_safe_barrier()

print('Imports OK | AllGather + barrier patched for single-process mode')

Imports OK | AllGather + barrier patched for single-process mode


In [3]:
CFG = SimpleNamespace(
    base_dir=str(BASE_DIR),
    output_dir=str(BASE_DIR / 'narvid_vn_output_nb'),
    init_model=None,
    resume_model=None,
    load_path=None,

    # training
    do_train=True,
    do_eval=False,
    eval_before_train=True,
    epochs=20,
    batch_size=32,
    batch_size_val=128,
    gradient_accumulation_steps=1,
    lr=1e-4,
    warmup_proportion=0.1,
    coef_lr=1.0,
    n_display=50,
    num_thread_reader=4,

    # NarVid config
    cross_model='cross-base',
    cache_dir='',
    task_type='retrieval',
    datatype='vietnamese',
    pretrained_clip_name='ViT-B/32',
    sim_header='meanP',
    loose_type=True,
    linear_patch='2d',
    cross_num_hidden_layers=4,
    max_words=32,
    max_frames=12,
    freeze_layer_num=6,

    # NarVid in-batch hard negative
    hard_negative_selection_factor=0.7,
    hard_negative_loss_factor=1.8,
    hard_negative_weighting=1.0,

    # Explicit hard negative from dataset negatives (train cell)
    explicit_hn_weight=0.5,
    explicit_hn_margin=0.05,

    nucleus_P=0.4,
    temperature=0.1,

    # single-GPU / non-DDP defaults required by NarVid internals
    local_rank=0,
    rank=0,
    world_size=1,
    distributed=False,
    n_gpu=1,
    seed=42,
)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(CFG.seed)
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
logger = get_logger(str(Path(CFG.output_dir) / 'log.txt'))
logger.info('Notebook config initialized')
print('output_dir:', CFG.output_dir)

04/16/2026 11:17:23 - INFO -   Notebook config initialized


output_dir: /media/urlab/KINGSTON/aic/narvid_vn_output_nb


In [4]:
def register_vietnamese_dataloader():
    DATALOADER_DICT['vietnamese'] = {
        'train': dataloader_vietnamese_train,
        'val': dataloader_vietnamese_test,
        'test': dataloader_vietnamese_test,
    }

def init_model(cfg, device):
    model_state_dict = None
    if cfg.init_model:
        model_state_dict = torch.load(cfg.init_model, map_location='cpu')

    cache_dir = cfg.cache_dir or os.path.join(str(PYTORCH_PRETRAINED_BERT_CACHE), 'distributed')
    model = NarVid.from_pretrained(
        cfg.cross_model,
        cache_dir=cache_dir,
        state_dict=model_state_dict,
        task_config=cfg,
    )
    model.to(device)
    return model

def load_model(cfg, device, model_file=None):
    ckpt = model_file or cfg.load_path
    if not ckpt:
        raise ValueError('No checkpoint file provided')
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f'Checkpoint not found: {ckpt}')

    state_dict = torch.load(ckpt, map_location='cpu')
    cache_dir = cfg.cache_dir or os.path.join(str(PYTORCH_PRETRAINED_BERT_CACHE), 'distributed')
    model = NarVid.from_pretrained(
        cfg.cross_model,
        cache_dir=cache_dir,
        state_dict=state_dict,
        task_config=cfg,
    )
    model.to(device)
    logger.info('Loaded checkpoint: %s', ckpt)
    return model

def freeze_clip_layers(model, cfg):
    assert -1 <= cfg.freeze_layer_num <= 12
    if not hasattr(model, 'clip') or cfg.freeze_layer_num <= -1:
        return

    for name, param in model.clip.named_parameters():
        if (
            name.startswith('ln_final.')
            or name.startswith('text_projection')
            or name.startswith('logit_scale')
            or name.startswith('visual.ln_post.')
            or name.startswith('visual.proj')
        ):
            continue

        if name.startswith('visual.transformer.resblocks.') or name.startswith('transformer.resblocks.'):
            layer_num = int(name.split('.resblocks.')[1].split('.')[0])
            if layer_num >= cfg.freeze_layer_num:
                continue

        if cfg.linear_patch == '3d' and 'conv2.' in name:
            continue

        param.requires_grad = False

def prep_optimizer(cfg, model, total_steps):
    named_params = list(model.named_parameters())
    no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']

    decay = [(n, p) for n, p in named_params if not any(nd in n for nd in no_decay)]
    nodecay = [(n, p) for n, p in named_params if any(nd in n for nd in no_decay)]

    decay_clip = [(n, p) for n, p in decay if 'clip.' in n]
    decay_noclip = [(n, p) for n, p in decay if 'clip.' not in n]
    nodecay_clip = [(n, p) for n, p in nodecay if 'clip.' in n]
    nodecay_noclip = [(n, p) for n, p in nodecay if 'clip.' not in n]

    weight_decay = 0.2
    grouped = [
        {'params': [p for _, p in decay_clip], 'weight_decay': weight_decay, 'lr': cfg.lr * cfg.coef_lr},
        {'params': [p for _, p in decay_noclip], 'weight_decay': weight_decay},
        {'params': [p for _, p in nodecay_clip], 'weight_decay': 0.0, 'lr': cfg.lr * cfg.coef_lr},
        {'params': [p for _, p in nodecay_noclip], 'weight_decay': 0.0},
    ]

    optimizer = BertAdam(
        grouped,
        lr=cfg.lr,
        warmup=cfg.warmup_proportion,
        schedule='warmup_cosine',
        b1=0.9,
        b2=0.98,
        e=1e-6,
        t_total=total_steps,
        weight_decay=weight_decay,
        max_grad_norm=1.0,
    )
    return optimizer, None

def save_model(epoch, cfg, model, optimizer, train_loss, type_name=''):
    suffix = '' if type_name == '' else f'{type_name}.'
    out_model = Path(cfg.output_dir) / f'pytorch_model.bin.{suffix}{epoch}'
    out_opt = Path(cfg.output_dir) / f'pytorch_opt.bin.{suffix}{epoch}'

    torch.save(model.state_dict(), out_model)
    torch.save({
        'epoch': epoch,
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': train_loss,
    }, out_opt)

    logger.info('Saved model: %s', out_model)
    logger.info('Saved optimizer: %s', out_opt)
    return str(out_model)

In [5]:
def _explicit_hard_negative_loss(
    model,
    input_ids, input_mask, segment_ids,
    video, video_mask,
    narration, narration_word_mask, narration_mask,
    neg_ids, neg_mask, neg_seg, neg_valid,
    margin: float = 0.05,
):
    """
    Explicit hard-negative loss from dataset negatives.
    Keeps base NarVid loss intact (square BxB), then adds hinge on provided negatives.
    """
    device = input_ids.device
    B = input_ids.shape[0]
    max_neg = neg_ids.shape[1]

    # [B*MAX_NEG] boolean mask of real negatives
    flat_valid = torch.cat([
        torch.arange(max_neg, device=device) < neg_valid[i]
        for i in range(B)
    ])

    if not flat_valid.any():
        return torch.tensor(0.0, device=device)

    owner_idx = torch.arange(B, device=device).unsqueeze(1).expand(B, max_neg).reshape(-1)[flat_valid]

    # Flatten negatives -> [N, 1, max_words]
    neg_ids_flat = neg_ids.view(B * max_neg, -1)[flat_valid].unsqueeze(1)
    neg_mask_flat = neg_mask.view(B * max_neg, -1)[flat_valid].unsqueeze(1)
    neg_seg_flat = neg_seg.view(B * max_neg, -1)[flat_valid].unsqueeze(1)

    # Positive features (B)
    seq_pos, word_pos, nar_pos, vis_pos = model.get_sequence_words_narration_visual_output(
        input_ids,
        segment_ids,
        input_mask,
        narration,
        narration_word_mask,
        narration_mask,
        video,
        video_mask,
        shaped=False,
    )

    tv_pos_c, tn_pos_c, tv_pos_f, tn_pos_f = model.get_similarity_logits(
        seq_pos, word_pos, vis_pos, nar_pos,
        input_mask, video_mask, narration_mask,
        shaped=False,
    )
    tv_pos = (tv_pos_c + tv_pos_f) / 2.0
    tn_pos = (tn_pos_c + tn_pos_f) / 2.0

    # Negative text features (N)
    seq_neg, word_neg = model.get_sequence_words_output(
        neg_ids_flat,
        neg_seg_flat,
        neg_mask_flat,
        shaped=False,
    )

    # Build masks in shaped=True format
    neg_text_mask = neg_mask_flat.view(-1, neg_mask_flat.shape[-1])       # [N, W]
    pos_video_mask = video_mask.view(-1, video_mask.shape[-1])            # [B, F]
    pos_narr_mask = narration_mask.view(-1, narration_mask.shape[-1])     # [B, F]

    tv_neg_c, tn_neg_c, tv_neg_f, tn_neg_f = model.get_similarity_logits(
        seq_neg, word_neg, vis_pos, nar_pos,
        neg_text_mask, pos_video_mask, pos_narr_mask,
        shaped=True,
    )
    tv_neg = (tv_neg_c + tv_neg_f) / 2.0   # [N, B]
    tn_neg = (tn_neg_c + tn_neg_f) / 2.0   # [N, B]

    row_idx = torch.arange(tv_neg.shape[0], device=device)
    neg_tv_anchor = tv_neg[row_idx, owner_idx]
    neg_tn_anchor = tn_neg[row_idx, owner_idx]

    pos_tv_anchor = tv_pos.diagonal()[owner_idx]
    pos_tn_anchor = tn_pos.diagonal()[owner_idx]

    # Want negatives lower than positives by margin
    loss_tv = torch.relu(neg_tv_anchor - pos_tv_anchor + margin)
    loss_tn = torch.relu(neg_tn_anchor - pos_tn_anchor + margin)
    return 0.5 * (loss_tv.mean() + loss_tn.mean())


def train_epoch(epoch, cfg, model, train_loader, device, optimizer, scheduler, global_step):
    model.train()
    total_loss = 0.0
    start_time = time.time()

    explicit_hn_weight = float(getattr(cfg, 'explicit_hn_weight', 0.5))
    explicit_hn_margin = float(getattr(cfg, 'explicit_hn_margin', 0.05))

    for step, batch in enumerate(train_loader):
        batch = tuple(t.to(device=device, non_blocking=True) for t in batch)
        input_ids, input_mask, segment_ids, video, video_mask, narration, narration_word_mask, narration_mask, \
            neg_ids, neg_mask, neg_seg, neg_valid = batch

        # Base NarVid loss already includes in-batch hard-negative mining.
        base_loss = model(input_ids, segment_ids, input_mask, video, video_mask, narration, narration_word_mask, narration_mask)

        # Extra explicit hard-negative loss from dataset negatives.
        explicit_hn_loss = _explicit_hard_negative_loss(
            model,
            input_ids, input_mask, segment_ids,
            video, video_mask,
            narration, narration_word_mask, narration_mask,
            neg_ids, neg_mask, neg_seg, neg_valid,
            margin=explicit_hn_margin,
        )

        loss = base_loss + explicit_hn_weight * explicit_hn_loss

        if cfg.gradient_accumulation_steps > 1:
            loss = loss / cfg.gradient_accumulation_steps

        loss.backward()
        total_loss += float(loss)

        if (step + 1) % cfg.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if scheduler is not None:
                scheduler.step()

            optimizer.step()
            optimizer.zero_grad()
            torch.clamp_(model.clip.logit_scale.data, max=np.log(100))

            global_step += 1
            if global_step % cfg.n_display == 0:
                lr_values = sorted(list(set(optimizer.get_lr())))
                logger.info(
                    'Epoch: %d/%d | Step: %d/%d | Lr: %s | Loss: %.6f | Base: %.6f | ExpHN: %.6f | Time/step: %.4f',
                    epoch + 1,
                    cfg.epochs,
                    step + 1,
                    len(train_loader),
                    '-'.join([f'{v:.9f}' for v in lr_values]),
                    float(loss),
                    float(base_loss),
                    float(explicit_hn_loss),
                    (time.time() - start_time) / (cfg.n_display * cfg.gradient_accumulation_steps),
                )
                start_time = time.time()

    return total_loss / max(1, len(train_loader)), global_step

In [6]:
def _run_on_single_gpu(model, batch_list_t, batch_list_v, batch_list_n, seq_outs, word_outs, vis_outs, nar_outs):
    sim_t2v_c, sim_t2n_c = [], []
    sim_t2v_f, sim_t2n_f = [], []

    for idx1, b1 in enumerate(batch_list_t):
        input_mask, segment_ids = b1
        sequence_output = seq_outs[idx1]
        word_output = word_outs[idx1]

        row_t2v_c, row_t2n_c = [], []
        row_t2v_f, row_t2n_f = [], []

        for idx2, (b2, b3) in enumerate(zip(batch_list_v, batch_list_n)):
            _ = idx2
            video_mask = b2[0]
            narration_mask = b3[0]
            visual_output = vis_outs[idx2]
            narration_output = nar_outs[idx2]

            l_t2v_c, l_t2n_c, l_t2v_f, l_t2n_f = model.get_similarity_logits(
                sequence_output,
                word_output,
                visual_output,
                narration_output,
                input_mask,
                video_mask,
                narration_mask,
            )

            row_t2v_c.append(l_t2v_c.detach().cpu().numpy())
            row_t2v_f.append(l_t2v_f.detach().cpu().numpy())
            row_t2n_c.append(l_t2n_c.detach().cpu().numpy())
            row_t2n_f.append(l_t2n_f.detach().cpu().numpy())

        sim_t2v_c.append(np.concatenate(tuple(row_t2v_c), axis=-1))
        sim_t2v_f.append(np.concatenate(tuple(row_t2v_f), axis=-1))
        sim_t2n_c.append(np.concatenate(tuple(row_t2n_c), axis=-1))
        sim_t2n_f.append(np.concatenate(tuple(row_t2n_f), axis=-1))

    return sim_t2v_c, sim_t2v_f, sim_t2n_c, sim_t2n_f

def get_score(tv_sim_matrix, tn_sim_matrix, multi_sentence, cut_off_points):
    tv_sim_matrix = (tv_sim_matrix - np.mean(tv_sim_matrix)) / (np.std(tv_sim_matrix) + 1e-8)
    tn_sim_matrix = (tn_sim_matrix - np.mean(tn_sim_matrix)) / (np.std(tn_sim_matrix) + 1e-8)

    t2v = v2t = tv_sim_matrix + tn_sim_matrix

    if multi_sentence:
        cut_off_plus = [x + 1 for x in cut_off_points]
        max_len = max([e - s for s, e in zip([0] + cut_off_plus[:-1], cut_off_plus)])

        t2v_new, v2t_new = [], []
        for s, e in zip([0] + cut_off_plus[:-1], cut_off_plus):
            t2v_new.append(np.concatenate((t2v[s:e], np.full((max_len - e + s, t2v.shape[1]), -np.inf)), axis=0))
            v2t_new.append(np.concatenate((v2t[s:e], np.full((max_len - e + s, v2t.shape[1]), -np.inf)), axis=0))

        t2v = np.stack(tuple(t2v_new), axis=0)
        v2t = np.stack(tuple(v2t_new), axis=0)

        tv_metrics = tensor_text_to_video_metrics(t2v)
        vt_metrics = compute_metrics(tensor_video_to_text_sim(v2t))
    else:
        tv_metrics = compute_metrics(t2v)
        vt_metrics = compute_metrics(v2t.T)

    logger.info('Text-to-Video: R1=%.2f R5=%.2f R10=%.2f MR=%.2f MeanR=%.2f', tv_metrics['R1'], tv_metrics['R5'], tv_metrics['R10'], tv_metrics['MR'], tv_metrics['MeanR'])
    logger.info('Video-to-Text: R1=%.2f R5=%.2f R10=%.2f MR=%.2f MeanR=%.2f', vt_metrics['R1'], vt_metrics['R5'], vt_metrics['R10'], vt_metrics['MR'], vt_metrics['MeanR'])

    return {
        'R1': float(tv_metrics['R1']),
        'R5': float(tv_metrics['R5']),
        'R10': float(tv_metrics['R10']),
        'MR': float(tv_metrics['MR']),
        'MeanR': float(tv_metrics['MeanR']),
        'SumR': float(tv_metrics['R1'] + tv_metrics['R5'] + tv_metrics['R10']),
    }

def eval_epoch(cfg, model, dataloader, device):
    if hasattr(dataloader.dataset, 'multi_sentence_per_video') and dataloader.dataset.multi_sentence_per_video:
        multi_sentence = True
        cut_off_points = [x - 1 for x in dataloader.dataset.cut_off_points]
    else:
        multi_sentence = False
        cut_off_points = []

    model.eval()
    with torch.no_grad():
        batch_list_t, batch_list_v, batch_list_n = [], [], []
        seq_outs, word_outs, vis_outs, nar_outs = [], [], [], []

        for bid, batch in enumerate(dataloader):
            batch = tuple(t.to(device) for t in batch)
            # Unpack only the first 8 tensors; ignore neg_* tensors (not needed for eval).
            input_ids, input_mask, segment_ids, video, video_mask, narration, narration_word_mask, narration_mask, *_ = batch

            sequence_output, word_output, narration_output, visual_output = model.get_sequence_words_narration_visual_output(
                input_ids,
                segment_ids,
                input_mask,
                narration,
                narration_word_mask,
                narration_mask,
                video,
                video_mask,
            )

            seq_outs.append(sequence_output)
            word_outs.append(word_output)
            vis_outs.append(visual_output)
            nar_outs.append(narration_output)
            batch_list_t.append((input_mask, segment_ids))
            batch_list_v.append((video_mask,))
            batch_list_n.append((narration_mask,))

            if (bid + 1) % 10 == 0 or (bid + 1) == len(dataloader):
                logger.info('Eval caching: %d/%d', bid + 1, len(dataloader))

        tv_c, tv_f, tn_c, tn_f = _run_on_single_gpu(model, batch_list_t, batch_list_v, batch_list_n, seq_outs, word_outs, vis_outs, nar_outs)
        tv_c = np.concatenate(tuple(tv_c), axis=0)
        tv_f = np.concatenate(tuple(tv_f), axis=0)
        tn_c = np.concatenate(tuple(tn_c), axis=0)
        tn_f = np.concatenate(tuple(tn_f), axis=0)

        tv_sim_matrix = (tv_c + tv_f) / 2.0
        tn_sim_matrix = (tn_c + tn_f) / 2.0

    return get_score(tv_sim_matrix, tn_sim_matrix, multi_sentence, cut_off_points)

def write_eval_summary(output_dir: str, baseline: dict, metrics: dict, split_name: str, stage: str):
    payload = {
        'timestamp': datetime.utcnow().isoformat() + 'Z',
        'stage': stage,
        'split': split_name,
        'baseline': baseline,
        'model': metrics,
        'delta_R1': metrics['R1'] - baseline['R1'],
        'delta_SumR': metrics['SumR'] - baseline['SumR'],
    }
    out = Path(output_dir) / f'eval_{stage}_{split_name}.json'
    out.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    logger.info('Saved eval summary: %s', out)
    return out

In [7]:
register_vietnamese_dataloader()
assert CFG.datatype in DATALOADER_DICT, f'Unsupported datatype: {CFG.datatype}'

tokenizer = ClipTokenizer()
model = init_model(CFG, DEVICE)
freeze_clip_layers(model, CFG)

train_loader, train_len, train_sampler = DATALOADER_DICT[CFG.datatype]['train'](CFG, tokenizer)
val_loader, val_len = DATALOADER_DICT[CFG.datatype]['val'](CFG, tokenizer, subset='val')
test_loader, test_len = DATALOADER_DICT[CFG.datatype]['test'](CFG, tokenizer, subset='test')

total_steps = (int(len(train_loader) + CFG.gradient_accumulation_steps - 1) / CFG.gradient_accumulation_steps) * CFG.epochs
optimizer, scheduler = prep_optimizer(CFG, model, total_steps)

baseline_path = SOURCE_DIR / 'narvid_baseline_results.json'
if baseline_path.exists():
    baseline_results = json.loads(baseline_path.read_text(encoding='utf-8'))
else:
    baseline_results = {'R1': 2.74, 'R5': 7.53, 'R10': 12.33, 'MR': 106.5, 'SumR': 22.60}

print('train_len:', train_len)
print('val_len  :', val_len)
print('test_len :', test_len)
print('baseline :', {k: baseline_results[k] for k in ['R1', 'R5', 'R10', 'MR', 'SumR']})

04/16/2026 11:17:24 - INFO -   loading archive file /media/urlab/KINGSTON/aic/narvid/modules/cross-base
04/16/2026 11:17:24 - INFO -   Model config {
  "attention_probs_dropout_prob": 0.1,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "max_position_embeddings": 128,
  "num_attention_heads": 8,
  "num_hidden_layers": 4,
  "type_vocab_size": 2,
  "vocab_size": 512
}

04/16/2026 11:17:24 - INFO -   Weight doesn't exsits. /media/urlab/KINGSTON/aic/narvid/modules/cross-base/cross_pytorch_model.bin
04/16/2026 11:17:24 - WARNING -   Stage-One:True, Stage-Two:False
04/16/2026 11:17:24 - WARNING -   Test retrieval by loose type.
04/16/2026 11:17:24 - WARNING -   	 embed_dim: 512
04/16/2026 11:17:24 - WARNING -   	 image_resolution: 224
04/16/2026 11:17:24 - WARNING -   	 vision_layers: 12
04/16/2026 11:17:24 - WARNING -   	 vision_width: 768
04/16/2026 11:17:24 - WARNING -   	 vision_patch_size: 32
04/16/2

[VietnameseNarVidDataset:train] skip K01_V031_000: Empty frames/transcripts for K01_V031_000
[VietnameseNarVidDataset:train] skip K19_V015_007: Empty frames/transcripts for K19_V015_007
[VietnameseNarVidDataset:val] skip K01_V031_000: Empty frames/transcripts for K01_V031_000
[VietnameseNarVidDataset:val] skip K19_V015_007: Empty frames/transcripts for K19_V015_007
[VietnameseNarVidDataset:test] skip K01_V031_000: Empty frames/transcripts for K01_V031_000
[VietnameseNarVidDataset:test] skip K19_V015_007: Empty frames/transcripts for K19_V015_007
train_len: 9222
val_len  : 9222
test_len : 9222
baseline : {'R1': 2.73972602739726, 'R5': 7.534246575342466, 'R10': 12.32876712328767, 'MR': 106.5, 'SumR': 22.602739726027394}


In [8]:
# Tạm thời bỏ eval_before_train trong notebook này.
print('Skipped pre-train evaluation (disabled in this notebook)')

Skipped pre-train evaluation (disabled in this notebook)


In [ ]:
best_r1 = -1.0
best_model_file = ''
global_step = 0

resumed_epoch = 0
if CFG.resume_model:
    checkpoint = torch.load(CFG.resume_model, map_location='cpu')
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    resumed_epoch = int(checkpoint['epoch']) + 1
    logger.info('Resumed optimizer from %s at epoch=%d', CFG.resume_model, resumed_epoch)

for epoch in range(resumed_epoch, CFG.epochs):
    if train_sampler is not None and hasattr(train_sampler, 'set_epoch'):
        train_sampler.set_epoch(epoch)

    train_loss, global_step = train_epoch(
        epoch,
        CFG,
        model,
        train_loader,
        DEVICE,
        optimizer,
        scheduler,
        global_step,
    )
    logger.info('Epoch %d/%d finished | train_loss=%.6f', epoch + 1, CFG.epochs, train_loss)

    ckpt_file = save_model(epoch, CFG, model, optimizer, train_loss)

    val_metrics = eval_epoch(CFG, model, val_loader, DEVICE)
    write_eval_summary(CFG.output_dir, baseline_results, val_metrics, split_name='val', stage=f'epoch_{epoch+1}')

    if val_metrics['R1'] > best_r1:
        best_r1 = val_metrics['R1']
        best_model_file = ckpt_file
        logger.info('New best model: %s | Val R@1=%.2f', best_model_file, best_r1)

print('Best checkpoint:', best_model_file)
print('Best Val R@1  :', best_r1)

/media/urlab/KINGSTON/aic/narvid/modules/modeling_narvid.py:786: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  text_weight.masked_fill_(torch.tensor((1 - word_mask), dtype=torch.bool), float("-inf"))
/tmp/ipykernel_42082/3629422789.py:120: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss += float(loss)
04/16/2026 11:18:28 - INFO -   Epoch: 1/20 | Step: 50/288 | Lr:  | Loss: 6.994119 | Base: 6.969086 | ExpHN: 0.050066 | Time/step: 1.1799
04/16/2026 11:19:25 - INFO -   Epoch: 1/20 | Step: 100/288 | Lr:  | Loss: 6.972318 | Base: 6.947402 | ExpHN: 0.049832 | Time/step: 1.1555
04/16/2026 11:20:23 - INFO -   Epoch: 1/20 | Step: 150/288 | Lr:  |

KeyboardInterrupt: 

: 

In [ ]:
if not best_model_file:
    print('No best model found. Run training cell first.')
else:
    best_model = load_model(CFG, DEVICE, best_model_file)
    test_metrics = eval_epoch(CFG, best_model, test_loader, DEVICE)
    out_file = write_eval_summary(CFG.output_dir, baseline_results, test_metrics, split_name='test', stage='best_model')

    compare = {
        'baseline_R1': baseline_results['R1'],
        'finetuned_R1': test_metrics['R1'],
        'delta_R1': test_metrics['R1'] - baseline_results['R1'],
        'baseline_SumR': baseline_results['SumR'],
        'finetuned_SumR': test_metrics['SumR'],
        'delta_SumR': test_metrics['SumR'] - baseline_results['SumR'],
    }

    print(json.dumps(compare, indent=2))
    print('Saved summary:', out_file)

04/16/2026 10:07:11 - INFO -   loading archive file /media/urlab/KINGSTON/aic/narvid/modules/cross-base
04/16/2026 10:07:11 - INFO -   Model config {
  "attention_probs_dropout_prob": 0.1,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 512,
  "initializer_range": 0.02,
  "intermediate_size": 2048,
  "max_position_embeddings": 128,
  "num_attention_heads": 8,
  "num_hidden_layers": 4,
  "type_vocab_size": 2,
  "vocab_size": 512
}

04/16/2026 10:07:11 - INFO -   Weight doesn't exsits. /media/urlab/KINGSTON/aic/narvid/modules/cross-base/cross_pytorch_model.bin
04/16/2026 10:07:11 - WARNING -   Stage-One:True, Stage-Two:False
04/16/2026 10:07:11 - WARNING -   Test retrieval by loose type.
04/16/2026 10:07:11 - WARNING -   	 embed_dim: 512
04/16/2026 10:07:11 - WARNING -   	 image_resolution: 224
04/16/2026 10:07:11 - WARNING -   	 vision_layers: 12
04/16/2026 10:07:11 - WARNING -   	 vision_width: 768
04/16/2026 10:07:11 - WARNING -   	 vision_patch_size: 32
04/16/2

{
  "baseline_R1": 2.73972602739726,
  "finetuned_R1": 0.3424657534246575,
  "delta_R1": -2.3972602739726026,
  "baseline_SumR": 22.602739726027394,
  "finetuned_SumR": 5.136986301369863,
  "delta_SumR": -17.46575342465753
}
Saved summary: /media/urlab/KINGSTON/aic/narvid_vn_output_nb/eval_best_model_test.json


: 

## Gợi ý sử dụng

1. Chạy lần lượt từ Cell 1 đến Cell 8 để setup.
2. Cell 9 để eval trước train (tuỳ chọn).
3. Cell 10 để train + val mỗi epoch.
4. Cell 11 để test checkpoint tốt nhất và in so sánh với baseline.

Notebook này không sửa file trong narvid, chỉ load class/hàm có sẵn.